# 1. Create Train-Test Split

We split 81 participants into 57 participants for training dataset and 24 participants for testing dataset

## 1.0 - Setup (As per Data Analysis notebook)

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

WORKING_DIR = Path.cwd()
DATA_DIR = WORKING_DIR.parent / "02_logs"

## 1.1 - Train Test Split

57 participants are assigned to the training/fit set and 24 participants are assigned to the held-out/test set.

In [2]:
from pathlib import Path

json_files = sorted(
    Path(DATA_DIR).rglob("*.json")
)

print("Total JSON Files: ", len(json_files))

Total JSON Files:  81


In [3]:
import random
random.seed(42)

shuffled_files = json_files.copy()
random.shuffle(shuffled_files)

train_files = shuffled_files[:57]
test_files = shuffled_files[57:]

print("Training files: ", len(train_files))
print("Testing files: ", len(test_files))

Training files:  57
Testing files:  24


In [4]:
train_set = set(train_files)
test_set = set(test_files)

print("Total Unique Files: ", len(train_set.union(test_set)))
print("Overlapping Files: ", len(train_set.intersection(test_set)))

Total Unique Files:  81
Overlapping Files:  0


In [5]:
train_file_list = [str(p) for p in train_files]
test_file_list = [str(p) for p in test_files]

with open("train_files.txt", "w", encoding = "utf-8") as f:
    f.write("\n".join(train_file_list))
    
with open("test_files.txt", "w", encoding = "utf-8") as f:
    f.write("\n".join(test_file_list))

Each participant has exactly 2 control counts (pure human written texts) and 4 treatment counts (human edited llm generated texts).

# 2 - Construct Human Style Reference

## 2.0 Theory

Each participant has 2 control texts (pure human written) and 4 treatment texts (Ai generated human edited).

We intend to generate an embedding from the participant's control texts as well as his treatment texts.

And then we would find LUAR similarity between the reference control text and his individual treatment texts. This would give us how much is the person able to bring an AI generated text into his own writing style.

How do we generate an embedding? We concatenate the two control texts, and then find its embedding.

Participant_id

control_text_1

control_text_2

control_reference



For each treatment i (Treatment 1, Treatment 2, Treatment 3 and Treatment 4), we would find 
1. LUAR embedding of AI draft (Lets call it AI_i)
2. LUAR embedding of final draft (Lets call it final_i)
3. Find Similarity between AI_i and control embedding
4. Find similarity between final_i and control embedding
5. Subtract the latter from the former 

if the difference is positive, it means that the participant's post edited text moved closer to their own control-writing representation.

### How does it happen in code?

1. Raw tezt
2. LUAR tokenizer
3. LUAR-MUD model
4. 512 dimensional vector

We are using the pretrained/frozen LUAR-MUD model, not training LUAR on our 81 participants. The official model was trained on the Reddit Million User Dataset

In [6]:
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "rrivera1849/LUAR-MUD"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model.eval()

print("LUAR loaded successfully!")

c:\Users\Samarth PC\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /rrivera1849/LUAR-MUD/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 0e0cd2d8-c463-49ad-81b5-f4f8f12bb7ae)')' thrown while requesting HEAD https://huggingface.co/rrivera1849/LUAR-MUD/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /rrivera1849/LUAR-MUD/resolve/main/tokenizer_config.json (Caused by NameResolutionError("HTTPSConne

LUAR loaded successfully!


## 2.1 - Embedding Generation testing for 1 participant

### 2.1.0 **METHODOLOGY**

I want to load LUAR model and generate embeddings - let us first do it for one participant. 

This includes :
1. 1 embedding of the AI text (how Ai writes - we concatenate 6 AI texts and find its embedding). 
2. Embeding of his pure human writing (concatenate 2 oure human writings and then find its embedding)
3. The 4 treatment embeddings - 1 each for the 4 treatment texts. 

Then I calculate and return the % to which the participant was able to get Ai generated Text A, B, C and D into his own writing. 

**This will be an entire function.**

### 2.1.1 - Function to generate LUAR Embedding
Generate a single 512-dimensional LUAR-MUD embedding for one text document.

#### Function to generate an LUAR embedding of a piece of writing

In [7]:
import torch

def luar_embedding(text, max_length=512):
    # Tokenize one document
    inputs = tokenizer(
        [text],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

    # LUAR expects:
    # (batch_size, episode_length, sequence_length)
    inputs["input_ids"] = inputs["input_ids"].reshape(1, 1, -1)
    inputs["attention_mask"] = inputs["attention_mask"].reshape(1, 1, -1)

    # No gradients needed for inference
    with torch.inference_mode():
        embedding = model(**inputs)

    # Shape: (1, 512) → (512,)
    return embedding.squeeze(0).cpu()

#### This is just to debug the luar_embedding function

In [8]:
import torch

def luar_embedding_see(text, max_length=512):
    # Tokenize one document
    inputs = tokenizer(
        [text],
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

    print("1", inputs)
    
    # LUAR expects:
    # (batch_size, episode_length, sequence_length)
    inputs["input_ids"] = inputs["input_ids"].reshape(1, 1, -1)
    inputs["attention_mask"] = inputs["attention_mask"].reshape(1, 1, -1)

    print("2", "-"*50, "\n")
    print(inputs)
    print("3", "-"*50, "\n")
    print(*inputs)
    print("5", "-"*50, "\n")
    
    # No gradients needed for inference
    with torch.inference_mode():
        embedding = model(**inputs)

    

    # Shape: (1, 512) → (512,)
    print(embedding.squeeze(0).cpu())

#### See the inputs passed and the final LUAR vector

In [9]:
with open(train_files[10], "r", encoding = "utf-8") as f:
    participant = json.load(f)

print(list(participant.keys()), "\n")

print(participant.get("user_info", {}).get("id"))
print(list(participant.get("responses", {}).keys()))
print(list(participant.get("responses", {}).get("0", {}).keys()), "\n\n")
print(participant.get("responses", {}).get("0", {}).get('final_version'), "\n\n\n")

test_ = participant.get("responses", {}).get("0", {}).get('final_version')
luar_embedding_see(test_, min(len(test_) + 1, 512))
# luar_embedding_see(test_, len(test_) + 1).shape)

['user_info', 'responses', 'pre_survey', 'mid_survey_1', 'mid_survey_2', 'post_survey'] 

f5d09877-f461-4169-a291-79c17218071d
['0', '1', '2', '3', '4', '5']
['start_time', 'scenario', 'model_generation_shown', 'details', 'model_generation', 'submit_details_time', 'final_version', 'edits', 'submit_final_text_time', 'likert'] 


Hello Paul! Hope you are doing great. I feel privileged to express my gratitude for what you did when i lost my mum. I can't imagine the financial support you offered me. Not forgeting the moral support you accorded me. By then my spirits were real low but you uplifted them. I can't give enough through my writing but you made me feel a very important person when i had just lost all the hopes. Thank you so much Paul for standing with me during that trying moment. I know i can accumulate enough to repay you for the hospitality but i felt honoured in your presence. You made me gain confident that had dwindled away. I really felt proud by your presence during that t

In [10]:
print(train_files[:10])

[WindowsPath('d:/Academics Semester 5/11 - DSL501 - Machine Learning/03 - Project/MLProject/02_logs/b417dba7-2b67-4496-b985-fc67c15aa748.json'), WindowsPath('d:/Academics Semester 5/11 - DSL501 - Machine Learning/03 - Project/MLProject/02_logs/32a9cad7-5b1c-480f-832c-1cedb782d226.json'), WindowsPath('d:/Academics Semester 5/11 - DSL501 - Machine Learning/03 - Project/MLProject/02_logs/99d7de62-94ed-4cc5-99bf-dbb5974df705.json'), WindowsPath('d:/Academics Semester 5/11 - DSL501 - Machine Learning/03 - Project/MLProject/02_logs/a03e9b46-d5c1-401e-ba35-3746d2880c76.json'), WindowsPath('d:/Academics Semester 5/11 - DSL501 - Machine Learning/03 - Project/MLProject/02_logs/6998773e-d1b4-4b6d-ac29-f75d132ecb89.json'), WindowsPath('d:/Academics Semester 5/11 - DSL501 - Machine Learning/03 - Project/MLProject/02_logs/21d20b0e-b89e-42a8-b0bb-cb8b2d2e35df.json'), WindowsPath('d:/Academics Semester 5/11 - DSL501 - Machine Learning/03 - Project/MLProject/02_logs/a1a47852-cad9-495d-a871-6797dc780d4f

### 2.1.2 - Participant Level Structure


#### **Theory**

Given a json file path = train_files[i] or test_files[j] or json_files[k]

1. We would do -> with open(path, "r", encoding = "utf-8") as f: data = json.load(f)
2. Go to data['responses']['i'] for each i = 0, 1, 2, 3, 4, 5
3. Find data['responses']['i']['model_generation'] and ['final_version']
4. Keep ['model_generation'] texts as a list **ai_texts**
5. Divide ['final_version'] by ['responses']['i']['model_generation_shown']
6. Store them in lists **human_texts** and **treatment_texts**.

7. Concatenate AI texts into string **ai_combined = "\n\n".join(ai_texts)**
8. Concatenate human texts into string **human_combined = "\n\n".join(human_texts)**

9. Find embeddings of them. **ai_embedding and human_embedding**.
10. List of human edited texts embeddings as **treatment_embeddings**.

#### Function to check progress of a post edited text from ai text towards human text

In [11]:
def progress_to_human(path, print_ans = False):

    with open(path, "r", encoding = "utf-8") as f:
        data = json.load(f)

    # Build the control (self-style) reference from the 2 pure human-written texts
    control_texts = []
    for i in range(6):
        response = data["responses"][str(i)]
        if response["model_generation_shown"] == 0:
            control_texts.append(response["final_version"])

    control_combined = "\n\n".join(control_texts)
    control_embedding = luar_embedding(control_combined, 512)

    # Per-task paired comparison: each AI draft and its own final edited
    # version are compared individually against the control reference,
    # instead of blending all 6 AI drafts into one truncated embedding.
    ai_sims = []
    final_sims = []

    for i in range(6):
        response = data["responses"][str(i)]
        if response["model_generation_shown"] == 1:
            ai_embedding_i = luar_embedding(response["model_generation"], 512)
            final_embedding_i = luar_embedding(response["final_version"], 512)

            ai_sims.append(cosine_sim(ai_embedding_i, control_embedding))
            final_sims.append(cosine_sim(final_embedding_i, control_embedding))

    changes = [f - a for f, a in zip(final_sims, ai_sims)]
    mean_change = sum(changes) / len(changes)

    if print_ans:
        print("AI draft vs control similarities: ", [x.item() for x in ai_sims])
        print("Final edited vs control similarities: ", [x.item() for x in final_sims])
        print("Changes:", [x.item() for x in changes])
        print("Mean change:", mean_change.item())

    return (
        [x.item() for x in ai_sims],
        [x.item() for x in final_sims],
        [x.item() for x in changes],
        mean_change.item()
    )

#### Function to find cosine similarity

In [12]:
def cosine_sim(a, b):
    dot_prd = torch.dot(a, b)
    den = torch.norm(a)*torch.norm(b)
    ans = dot_prd/den
    return ans

Check for one participant

In [13]:
ai_sims, final_sims, changes, mean_change = progress_to_human(train_files[0], print_ans=True)

print("AI-draft vs control similarities:", ai_sims)
print("Final-edited vs control similarities:", final_sims)
print("Changes:", changes)
print("Mean change:", mean_change)

AI draft vs control similarities:  [0.5335302352905273, 0.564225971698761, 0.6228578686714172, 0.658645749092102]
Final edited vs control similarities:  [0.723385751247406, 0.5976560711860657, 0.6870607137680054, 0.6753390431404114]
Changes: [0.18985551595687866, 0.03343009948730469, 0.06420284509658813, 0.016693294048309326]
Mean change: 0.0760454386472702
AI-draft vs control similarities: [0.5335302352905273, 0.564225971698761, 0.6228578686714172, 0.658645749092102]
Final-edited vs control similarities: [0.723385751247406, 0.5976560711860657, 0.6870607137680054, 0.6753390431404114]
Changes: [0.18985551595687866, 0.03343009948730469, 0.06420284509658813, 0.016693294048309326]
Mean change: 0.0760454386472702


Function to check in all 81 participants

In [14]:
df = pd.DataFrame(columns = ["participant_id", "ai_control_sims", "final_control_sims", "changes", "mean_change"])

for prtc in train_files:
    ai_sims, final_sims, changes, mean_change = progress_to_human(prtc)
    df.loc[len(df)] = [prtc, ai_sims, final_sims, changes, mean_change]

Save it in a csv file

In [15]:
saving_path = WORKING_DIR / "personal_explore" / "6j28b_train_similarity_paired.csv"
saving_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(saving_path, index=False)
print("Saved to:", saving_path)

Saved to: d:\Academics Semester 5\11 - DSL501 - Machine Learning\03 - Project\MLProject\task_A3\personal_explore\6j28b_train_similarity_paired.csv


## 2.2 - Statistical Analysis

In [16]:
df["mean_change"].describe()

count    57.000000
mean      0.053909
std       0.046255
min      -0.003938
25%       0.019644
50%       0.044816
75%       0.070422
max       0.188457
Name: mean_change, dtype: float64

In [17]:
print("Mean:", df["mean_change"].mean())
print("Median:", df["mean_change"].median())
print("Positive:", (df["mean_change"] > 0).sum())
print("Negative:", (df["mean_change"] < 0).sum())

Mean: 0.053909498087146825
Median: 0.044815920293331146
Positive: 55
Negative: 2


Our assessment is showing that post human editing is moving away from the human reference on average.


In [18]:
df[["change_1", "change_2", "change_3", "change_4"]] = pd.DataFrame(
    df["changes"].tolist(),
    index=df.index
)

In [19]:
df[["change_1", "change_2", "change_3", "change_4"]].describe()

,change_1,change_2,change_3,change_4
count,57.000000,57.000000,57.000000,57.000000
mean,0.061736,0.048803,0.055943,0.049156
std,0.073349,0.058227,0.057490,0.062648
min,-0.029967,-0.037771,-0.010901,-0.064420
25%,0.005307,0.001312,0.005645,0.000000
50%,0.040342,0.035547,0.035133,0.034713
75%,0.087369,0.071589,0.087635,0.079186
max,0.269106,0.233790,0.207504,0.244962


#### Why our result was negative (resolved)

The original `progress_to_human` built a single "AI voice" embedding by concatenating all 6 `model_generation` texts (~5000 characters) and truncating to LUAR's 512-token limit — this silently dropped most of the drafts and mixed in texts the participant never even saw (the 2 "pure" condition drafts). That blended, truncated embedding was then compared against each individual, untruncated treatment text, producing a biased, non-paired comparison.

We fixed this by comparing each treatment task's own AI draft and its own final edited version, individually, against the control reference (§2.1.1 cell `22747a3f`). After the fix, the train-set result flipped from mean change **-0.026** (55/57 negative) to **+0.054** (55/57 positive) — now directionally consistent with the paper's H1a.

# 3. Held-Out Validation (Module A3)

The train-set result above (§2.2) was fit and inspected on the 57 training participants. Module A3's actual deliverable is to check whether the same effect replicates on the 24 **held-out, participant-disjoint** test participants that were set aside in §1.1 and never touched until now.

We reuse the exact same `progress_to_human` function (no refitting, nothing calibrated on test data — LUAR-MUD is a frozen pretrained model) and simply run it on `test_files` instead of `train_files`.

## 3.1 - Run the paired comparison on the 24 held-out participants

In [20]:
test_df = pd.DataFrame(columns = ["participant_id", "ai_control_sims", "final_control_sims", "changes", "mean_change"])

for prtc in test_files:
    ai_sims, final_sims, changes, mean_change = progress_to_human(prtc)
    test_df.loc[len(test_df)] = [prtc, ai_sims, final_sims, changes, mean_change]

print("Held-out participants processed:", len(test_df))

Held-out participants processed: 24


In [21]:
test_saving_path = WORKING_DIR / "personal_explore" / "test_similarity_paired.csv"
test_saving_path.parent.mkdir(parents=True, exist_ok=True)

test_df.to_csv(test_saving_path, index=False)
print("Saved to:", test_saving_path)

Saved to: d:\Academics Semester 5\11 - DSL501 - Machine Learning\03 - Project\MLProject\task_A3\personal_explore\test_similarity_paired.csv


## 3.2 - Statistical Analysis (Held-Out Set)

In [22]:
test_df["mean_change"].describe()

count    24.000000
mean      0.044878
std       0.043480
min      -0.001763
25%       0.005954
50%       0.037211
75%       0.066482
max       0.145048
Name: mean_change, dtype: float64

In [23]:
print("Mean:", test_df["mean_change"].mean())
print("Median:", test_df["mean_change"].median())
print("Positive:", (test_df["mean_change"] > 0).sum())
print("Negative:", (test_df["mean_change"] < 0).sum())

Mean: 0.04487812581161658
Median: 0.037210505455732346
Positive: 22
Negative: 1


In [24]:
test_df[["change_1", "change_2", "change_3", "change_4"]] = pd.DataFrame(
    test_df["changes"].tolist(),
    index=test_df.index
)

test_df[["change_1", "change_2", "change_3", "change_4"]].describe()

,change_1,change_2,change_3,change_4
count,24.000000,24.000000,24.000000,24.000000
mean,0.049370,0.047244,0.031814,0.051085
std,0.071092,0.059767,0.042727,0.059095
min,-0.028806,-0.008802,-0.008954,-0.001640
25%,0.000416,0.006938,0.000000,0.000000
50%,0.025366,0.026903,0.026351,0.029059
75%,0.066825,0.063911,0.050358,0.076462
max,0.324087,0.211380,0.177896,0.188448


## 3.3 - Train vs. Held-Out Comparison

Does the positive effect direction found on the 57 training participants (§2.2) replicate on the 24 held-out participants?

In [25]:
from scipy import stats

train_changes = df["mean_change"]
test_changes = test_df["mean_change"]

summary = pd.DataFrame({
    "train (n=57)": train_changes.describe(),
    "held-out (n=24)": test_changes.describe()
})
display(summary)

# One-sample Wilcoxon signed-rank test: is the mean_change significantly != 0?
train_stat, train_p = stats.wilcoxon(train_changes)
test_stat, test_p = stats.wilcoxon(test_changes)

print("\nWilcoxon signed-rank test (H0: median change = 0)")
print(f"  Train:     statistic={train_stat:.3f}, p={train_p:.6f}")
print(f"  Held-out:  statistic={test_stat:.3f}, p={test_p:.6f}")

# Effect size (matched-pairs rank-biserial correlation, reported alongside the paper's own Hedges' g style reporting)
def rank_biserial(x):
    x = np.asarray(x)
    pos = (x > 0).sum()
    neg = (x < 0).sum()
    return (pos - neg) / (pos + neg)

print("\nRank-biserial effect size")
print(f"  Train:     {rank_biserial(train_changes):.3f}")
print(f"  Held-out:  {rank_biserial(test_changes):.3f}")

,train (n=57),held-out (n=24)
count,57.000000,24.000000
mean,0.053909,0.044878
std,0.046255,0.043480
min,-0.003938,-0.001763
25%,0.019644,0.005954
50%,0.044816,0.037211
75%,0.070422,0.066482
max,0.188457,0.145048



Wilcoxon signed-rank test (H0: median change = 0)
  Train:     statistic=5.000, p=0.000000
  Held-out:  statistic=3.000, p=0.000040

Rank-biserial effect size
  Train:     0.930
  Held-out:  0.913


## 3.4 - Concrete Walkthrough: One Held-Out Participant

The aggregate numbers in §3.1-3.3 are easy to write off as abstract. To make them tangible, we pick one held-out participant (the one whose `mean_change` is closest to the held-out median — i.e. a "typical" participant, not a cherry-picked best case) and show:

1. **Direct text-to-text similarity** (no control reference involved): AI draft vs. its own final edited text, for the 4 treatment tasks.
2. **Direct text-to-text similarity**: the AI draft that was generated but never shown vs. the participant's own from-scratch writing, for the 2 pure tasks.
3. This participant's actual §3.1 control-referenced numbers (`ai_control_sims`, `final_control_sims`, `changes`, `mean_change`).
4. Where this participant's `mean_change` sits inside the held-out distribution from §3.2/§3.3.

In [26]:
median_val = test_df["mean_change"].median()
example_idx = (test_df["mean_change"] - median_val).abs().idxmin()
example_row = test_df.loc[example_idx]
example_path = example_row["participant_id"]

print("Chosen held-out participant (closest to held-out median mean_change):")
print(example_path)
print("mean_change:", round(example_row["mean_change"], 4), " | held-out median:", round(median_val, 4))

Chosen held-out participant (closest to held-out median mean_change):
d:\Academics Semester 5\11 - DSL501 - Machine Learning\03 - Project\MLProject\02_logs\fd93ee7a-8e03-4ead-8a80-9847e589e01e.json
mean_change: 0.027  | held-out median: 0.0372


In [27]:
with open(example_path, "r", encoding="utf-8") as f:
    example_data = json.load(f)

print("Participant:", example_data["user_info"]["id"])
print()

treatment_direct_sims = []
pure_direct_sims = []

for i in range(6):
    response = example_data["responses"][str(i)]
    ai_emb = luar_embedding(response["model_generation"], 512)
    final_emb = luar_embedding(response["final_version"], 512)
    direct_sim = cosine_sim(ai_emb, final_emb).item()

    if response["model_generation_shown"] == 1:
        treatment_direct_sims.append((response["scenario"], direct_sim))
    else:
        pure_direct_sims.append((response["scenario"], direct_sim))

print("1) AI draft vs. Human-edited final text (4 treatment tasks):")
for scenario, sim in treatment_direct_sims:
    print(f"   {scenario:12s}: {sim:.4f}")

print("\n2) AI draft (generated but never shown) vs. Pure human-written text (2 control tasks):")
for scenario, sim in pure_direct_sims:
    print(f"   {scenario:12s}: {sim:.4f}")

Participant: fd93ee7a-8e03-4ead-8a80-9847e589e01e

1) AI draft vs. Human-edited final text (4 treatment tasks):
   condolence  : 0.9647
   thank       : 0.9241
   letter      : 0.9888
   speech      : 0.9980

2) AI draft (generated but never shown) vs. Pure human-written text (2 control tasks):
   apology     : 0.4179
   wedding     : 0.5479


**Reading these two groups:** group (1) is a literal, un-mediated comparison — how much of the post-edited text still textually resembles the exact AI draft it came from. Group (2) is a topic-matched baseline — how similar an AI draft is to the participant's own unassisted writing on the *same scenario*, even though that draft was never seen or used. Group (1) being higher than group (2) is expected (the final text literally derives from editing the AI draft). Neither of these two groups is what Module A3 tests, though — they don't involve the control reference. The numbers that actually test "did editing move the text toward this person's *own* style" are the control-referenced ones below, from §3.1.

In [28]:
print("This participant's §3.1 control-referenced numbers (already computed in test_df):\n")
print("AI draft vs control similarities:    ", [round(x, 4) for x in example_row["ai_control_sims"]])
print("Final edited vs control similarities:", [round(x, 4) for x in example_row["final_control_sims"]])
print("Changes (final - ai):                ", [round(x, 4) for x in example_row["changes"]])
print("Mean change:                         ", round(example_row["mean_change"], 4))

This participant's §3.1 control-referenced numbers (already computed in test_df):

AI draft vs control similarities:     [0.6062, 0.405, 0.4623, 0.4325]
Final edited vs control similarities: [0.6214, 0.4413, 0.4989, 0.4524]
Changes (final - ai):                 [0.0152, 0.0364, 0.0365, 0.0199]
Mean change:                          0.027


In [29]:
rank = (test_df["mean_change"] < example_row["mean_change"]).sum() + 1
percentile = stats.percentileofscore(test_df["mean_change"], example_row["mean_change"])

print(f"Held-out set (§3.2) mean:   {test_df['mean_change'].mean():.4f}")
print(f"Held-out set (§3.2) median: {test_df['mean_change'].median():.4f}")
print(f"This participant:           {example_row['mean_change']:.4f}")
print(f"Rank: {rank} of {len(test_df)} held-out participants (sorted ascending)")
print(f"Percentile: {percentile:.1f}%")
print()
print(f"This participant's change is positive, so they are one of the "
      f"{(test_df['mean_change'] > 0).sum()}/{len(test_df)} participants driving the "
      f"§3.3 Wilcoxon result (p={test_p:.6f}, rank-biserial={rank_biserial(test_changes):.3f}).")

Held-out set (§3.2) mean:   0.0449
Held-out set (§3.2) median: 0.0372
This participant:           0.0270
Rank: 12 of 24 held-out participants (sorted ascending)
Percentile: 50.0%

This participant's change is positive, so they are one of the 22/24 participants driving the §3.3 Wilcoxon result (p=0.000040, rank-biserial=0.913).


## 3.5 - Effect Size: Hedges' g (comparable to the paper's own reporting)

Raw cosine-similarity deltas (§3.1-3.4) are hard to read on their own since they're scale-dependent. The source paper reports effect sizes using Hedges' g (e.g. g = -1.43 for H1c), so we compute the same standardized, small-sample-corrected effect size here for direct comparability — this also lets us state precisely how large the self-similarity increase is, not just that it's "positive."

In [30]:
def hedges_g(x):
    """One-sample Hedges' g: standardized mean_change against 0, with small-sample bias correction."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    mean = x.mean()
    sd = x.std(ddof=1)
    d = mean / sd
    correction = 1 - (3 / (4 * (n - 1) - 1))
    return d * correction

train_g = hedges_g(df["mean_change"])
test_g = hedges_g(test_df["mean_change"])

print("Hedges' g (one-sample, mean_change vs. 0)")
print(f"  Train (n=57):     g = {train_g:.3f}")
print(f"  Held-out (n=24):  g = {test_g:.3f}")
print()
print("For reference, the source paper reports Hedges' g = -1.43 for H1c")
print("(post-edited-to-LLM similarity vs. post-edited-to-control similarity).")
print("Conventional thresholds: |g| ~ 0.2 small, 0.5 medium, 0.8 large.")

Hedges' g (one-sample, mean_change vs. 0)
  Train (n=57):     g = 1.150
  Held-out (n=24):  g = 0.998

For reference, the source paper reports Hedges' g = -1.43 for H1c
(post-edited-to-LLM similarity vs. post-edited-to-control similarity).
Conventional thresholds: |g| ~ 0.2 small, 0.5 medium, 0.8 large.


### Conclusion

If the held-out p-value is small and the effect size/sign matches the training set, this is evidence that the positive post-editing effect (self-similarity increasing after editing, consistent with the paper's H1a) is **not an artifact of the 57 fit-set participants** and generalizes to unseen participants — which is the robustness check Module A3 set out to perform.